# KION AutoFE: полный прогон на Kaggle
Settings: **Accelerator = GPU T4**, **Internet = On**. Запуск: *Save Version → Save & Run All* — идёт в фоне, результаты в Output (`results/`, `logs/`).

In [ ]:
!nvidia-smi -L; nproc; free -g | head -2
!git clone -q https://github.com/r3dmistyle/kion-autofe.git /tmp/kion-autofe
%cd /tmp/kion-autofe
!pip -q install duckdb catboost tabulate
# results/ and logs/ live in /kaggle/working, so they are saved as notebook output even if a later cell fails
!mkdir -p /kaggle/working/results /kaggle/working/logs && ln -s /kaggle/working/results results && ln -s /kaggle/working/logs logs
!python tests/test_leakage.py && python src/ranker.py && python src/search.py

In [ ]:
# main run: baseline, hand-crafted, AutoFE search, comparison, window sweep
!python src/experiments.py --n-users 100000 --device gpu --seeds 3 --search-users 6000 --pool 100 --mut-fits 60 2>&1 | tee logs/n100000.log | grep -vE 'Warn|warn|df.n.|^screen'

In [ ]:
# scaling: same pipeline, features selected at 100k reused
!for n in 25000 50000 200000; do python src/experiments.py --n-users $n --device gpu --seeds 1 --steps e0,hand,auto --search-from results/n100000 2>&1 | tee logs/n$n.log | tail -4; done

In [ ]:
!python src/experiments.py --steps report && cat results/summary.md